IMPORTS

In [1]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

sys.path.append('../utils')
from utils import save_dataframe

CONFIGURATIONS

In [2]:
# ─────────────────────────────────────────
# CAMERA CONFIG
# Change CAMERA_SOURCE to 0 when real USB
# webcam is connected
# ─────────────────────────────────────────

# 0 = first USB webcam on your PC
# 'path/to/video.mp4' = simulate with video file
CAMERA_SOURCE = 'data/videos/sample_water.mp4'  # simulation mode

FRAME_INTERVAL = 30      # capture every 30 frames
MAX_FRAMES = 20          # max frames to capture per session
SAVE_FOLDER = 'data/frames'

os.makedirs(SAVE_FOLDER, exist_ok=True)
os.makedirs('data/videos', exist_ok=True)

print("Config ready!")
print(f"Camera source: {CAMERA_SOURCE}")
print(f"Saving frames to: {SAVE_FOLDER}")

Config ready!
Camera source: data/videos/sample_water.mp4
Saving frames to: data/frames


Download a sample water video for simulation

In [3]:
import requests

# Free sample video from Wikimedia
VIDEO_URL = "https://upload.wikimedia.org/wikipedia/commons/transcoded/8/8e/Muddy_river.ogv/Muddy_river.ogv.360p.webm"
VIDEO_PATH = "data/videos/sample_water.mp4"

if not os.path.exists(VIDEO_PATH):
    print("Downloading sample video...")
    response = requests.get(VIDEO_URL, stream=True, timeout=30)
    with open(VIDEO_PATH, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded: {VIDEO_PATH}")
else:
    print(f"Video already exists: {VIDEO_PATH}")

Downloaded: data/videos/sample_water.mp4


 Frame capture function

In [4]:
def capture_frames(source, frame_interval=30, max_frames=20, save_folder='data/frames'):
    """
    Capture frames from a video file or live webcam.
    
    source: 0 for USB webcam, or path to video file
    frame_interval: capture one frame every N frames
    max_frames: stop after this many captures
    save_folder: where to save the captured frames
    """
    cap = cv2.VideoCapture(source)
    
    if not cap.isOpened():
        print("Could not open video source.")
        return []

    captured = []
    frame_count = 0
    saved_count = 0

    print(f"Starting capture from: {source}")

    while saved_count < max_frames:
        ret, frame = cap.read()

        if not ret:
            print("End of video or no frame received.")
            break

        if frame_count % frame_interval == 0:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
            filename = f"frame_{timestamp}.jpg"
            path = os.path.join(save_folder, filename)

            cv2.imwrite(path, frame)
            captured.append({
                'filename': filename,
                'path': path,
                'frame_number': frame_count,
                'timestamp': timestamp
            })
            saved_count += 1
            print(f"Captured frame {saved_count}: {filename}")

        frame_count += 1

    cap.release()
    print(f"\nDone! Captured {saved_count} frames.")
    return captured

Run the capture

In [5]:
# Run capture using simulation video
# When real webcam is ready, change CAMERA_SOURCE = 0
frames_data = capture_frames(
    source=CAMERA_SOURCE,
    frame_interval=FRAME_INTERVAL,
    max_frames=MAX_FRAMES,
    save_folder=SAVE_FOLDER
)

Could not open video source.


 Preview captured frames

In [ ]:
# Show first 6 captured frames in a grid
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < len(frames_data):
        img = cv2.imread(frames_data[i]['path'])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"Frame {i+1}", fontsize=10)
        ax.axis('off')
    else:
        ax.axis('off')

plt.suptitle('Captured Frames from Video Feed', fontsize=14)
plt.tight_layout()
plt.show()

Save frame metadata

In [ ]:
import pandas as pd

df_frames = pd.DataFrame(frames_data)
save_dataframe(df_frames, 'frame_metadata.csv')
print(df_frames.head())

In [ ]:
# Simulation (now)
CAMERA_SOURCE = 'data/videos/sample_water.mp4'

# Real webcam (later) — just change this one line!
# CAMERA_SOURCE = 0